<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/evaluation/01_baseline_performance_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================
# [Evaluation] Model Performance Test Environment Setup
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 평가 환경 설정을 시작합니다...")

# 1. Google Drive 마운트
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. GitHub 최신화 및 경로 설정
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# 3. 커스텀 모듈 실행 및 필수 패키지 설치
try:
    from src.env_setup import init_colab_env
    init_colab_env()
except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")

# 평가 및 비동기 처리용 필수 패키지 강제 설치
!pip install -q soundfile pyyaml tqdm museval mir_eval pretty_midi nest_asyncio pandas

import nest_asyncio
nest_asyncio.apply() # Colab(Jupyter) 내부 이벤트 루프 충돌 방지

print("\n🎉 Ready to Rock! 평가 환경 셋업 완료.")

In [ ]:
# ==================================================================
# [Data Load] 평가 데이터셋 Colab 로컬 디스크로 전송
# ==================================================================
import shutil
from pathlib import Path

# 경로 설정 (드라이브 내 slakh_eval 폴더 위치에 맞게 수정하세요)
DRIVE_EVAL_DIR = "/content/drive/MyDrive/Bass_separator/datasets/slakh_eval"
LOCAL_EVAL_DIR = "/content/slakh_eval"

if not os.path.exists(LOCAL_EVAL_DIR):
    print("📥 구글 드라이브에서 평가용 데이터셋을 Colab 로컬로 복사합니다 (속도 최적화)...")
    shutil.copytree(DRIVE_EVAL_DIR, LOCAL_EVAL_DIR)
    print("✅ 복사 완료!")
else:
    print("✅ 로컬에 이미 데이터가 존재합니다.")

track_dirs = [d for d in Path(LOCAL_EVAL_DIR).iterdir() if d.is_dir()]
print(f"🔍 총 {len(track_dirs)}개의 평가용 트랙이 감지되었습니다.")

In [ ]:
# ==================================================================
# [Sanity Check] 단일 트랙 채보 알고리즘 격리 평가 (Isolated Mode)
# ==================================================================
import asyncio
from src.evaluation import run_transcription_evaluation

# 테스트할 첫 번째 트랙 지정
sample_track = track_dirs[0]
ref_midi = str(sample_track / "bass_gt.mid")
audio_input = str(sample_track / "bass_gt.wav")  # Isolated: 정답 베이스 음원 직접 입력

print(f"🎵 테스트 트랙: {sample_track.name}")

async def test_single_track():
    metrics = await run_transcription_evaluation(
        ref_midi_path=ref_midi,
        audio_path=audio_input,
        is_isolated=True,         # Demucs 생략, 채보 성능만 순수 검증
        onset_tolerance=0.1       # 베이스 물리적 특성 반영 (100ms)
    )
    return metrics

sample_result = asyncio.run(test_single_track())
print("\n📊 산출된 평가 지표:")
for k, v in sample_result.items():
    print(f"  - {k}: {v}")

In [ ]:
# ==================================================================
# [Batch Evaluation] 전체 데이터셋 성능 평가 및 결과 저장
# ==================================================================
import pandas as pd
from tqdm.notebook import tqdm

# 설정 파라미터
RUN_E2E = False       # True: mix.wav 사용 (Demucs 포함 전체 파이프라인), False: bass_gt.wav 사용 (채보만 평가)
ONSET_TOLERANCE = 0.1 # 100ms
NUM_TRACKS_TO_TEST = 10 # 전체를 다 돌리려면 len(track_dirs) 로 변경하세요.

results = []
test_subset = track_dirs[:NUM_TRACKS_TO_TEST]

print(f"🚀 총 {len(test_subset)}개 트랙에 대한 배치 평가를 시작합니다. (E2E 모드: {RUN_E2E})")

async def evaluate_batch():
    for track in tqdm(test_subset, desc="Evaluating Tracks"):
        ref_midi = str(track / "bass_gt.mid")

        # E2E 모드면 mix.wav(원본 믹스), Isolated 모드면 bass_gt.wav(분리된 베이스) 사용
        input_audio = str(track / "mix.wav") if RUN_E2E else str(track / "bass_gt.wav")

        # 파일 존재 여부 검사
        if not os.path.exists(ref_midi) or not os.path.exists(input_audio):
            print(f"⚠️ 건너뜀 (파일 누락): {track.name}")
            continue

        try:
            metrics = await run_transcription_evaluation(
                ref_midi_path=ref_midi,
                audio_path=input_audio,
                is_isolated=not RUN_E2E,
                onset_tolerance=ONSET_TOLERANCE
            )

            # 메타데이터 추가
            metrics['Track_ID'] = track.name
            metrics['Mode'] = "E2E" if RUN_E2E else "Isolated"
            results.append(metrics)

        except Exception as e:
            print(f"❌ 오류 발생 ({track.name}): {e}")

# 비동기 실행
asyncio.run(evaluate_batch())

# -----------------------------------------------------------------
# 평가 결과 DataFrame 변환 및 저장
# -----------------------------------------------------------------
if results:
    df_results = pd.DataFrame(results)
    # 컬럼 순서 정리 (Track_ID가 맨 앞으로 오게)
    cols = ['Track_ID', 'Mode'] + [c for c in df_results.columns if c not in ['Track_ID', 'Mode']]
    df_results = df_results[cols]

    # 드라이브에 결과 저장
    RESULT_CSV_PATH = "/content/drive/MyDrive/Bass_separator/datasets/evaluation_results.csv"
    df_results.to_csv(RESULT_CSV_PATH, index=False)

    print("\n✅ 배치 평가 완료!")
    print(f"💾 평가 결과가 구글 드라이브에 저장되었습니다: {RESULT_CSV_PATH}")

    # 평균 점수 요약 출력
    print("-" * 40)
    print("🔹 전체 평균 지표 (Average Metrics)")
    print("-" * 40)
    avg_f1 = df_results['Onset_Pitch_F1'].mean() * 100
    avg_precision = df_results['Onset_Pitch_Precision'].mean() * 100
    avg_recall = df_results['Onset_Pitch_Recall'].mean() * 100

    print(f"🎯 Average Onset-Pitch F1     : {avg_f1:.2f}%")
    print(f"🎯 Average Onset-Pitch Precision: {avg_precision:.2f}%")
    print(f"🎯 Average Onset-Pitch Recall   : {avg_recall:.2f}%")
else:
    print("⚠️ 유효한 평가 결과가 없습니다.")